# LawLens, Module 1: judgment summariser

Run the cells **top to bottom**. Before you start: *Runtime > Change runtime type > T4 GPU*.

Each cell says what it does. Cells marked **(once)** only need to run once per session.

## 1. Get the code (once)
Pick **A or B**, not both.

**A. From GitHub** (after you have pushed the project). Put your own repo URL below.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/LawLens.git"   # <- change this
import os
%cd /content
!rm -rf LawLens
!git clone {REPO_URL}
%cd /content/LawLens

**B. From the zip** (before GitHub). First upload `LawLens_module1.zip` using the folder icon on the left.

In [ ]:
%cd /content
!rm -rf LawLens
!unzip -q LawLens_module1.zip
%cd /content/LawLens

## 2. Install (once)

In [ ]:
!pip install -q -r requirements.txt

## 3. Check the code works (no downloads needed)
Expect `16 passed`. Then a quick summary of the built-in sample judgment.

In [ ]:
!python -m pytest -q tests
!python scripts/summarise_cli.py samples/sample_judgment.txt

## 4. Download the IN-Abs dataset (once)
This pulls the public dataset (about 98 MB) from Zenodo straight into Colab. You do **not** need to download or upload any dataset yourself.

In [ ]:
!mkdir -p data/inabs
!wget -q -O data/dataset.zip "https://zenodo.org/records/7152317/files/dataset.zip?download=1"
!unzip -q -o data/dataset.zip -d data/inabs
!find data/inabs -maxdepth 4 -type d | head -30

If the next step cannot find the folders, copy the folder list printed above and send it to me.

## 5. Evaluate (reproduces the README table)
Settings were tuned on `train` and frozen: weights `0.3,0,0.7` (cue, embedding, position). Run 5a to see the check on 50 other training documents, then 5b for the final test numbers. Results are saved to `results/`.

In [ ]:
# 5a. validation on 50 training documents (seed 1)
!python scripts/eval_inabs.py --root data/inabs --split train --n 50 --seed 1 --max_words 450,900 --systems lead,cues --weights 0.3,0,0.7 --bert_model distilroberta-base --out results/validate_seed1

In [ ]:
# 5b. final run: 100 test documents
!python scripts/eval_inabs.py --root data/inabs --split test --max_words 450,900 --systems lead,cues --weights 0.3,0,0.7 --bert_model distilroberta-base --out results/module1_inabs
print(open("results/module1_inabs.md").read())

## 6. Run the app
This starts Streamlit and gives you a public link. Open the printed `https://....loca.lt` URL; if it asks for a password, use the IP address printed just above it.

In [ ]:
!npm install -s localtunnel
import subprocess, time, urllib.request
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
                 stdout=open("/content/streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)
print("Tunnel password (IP):", urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip())
!npx localtunnel --port 8501

## 7. Save your results
Colab wipes everything when the session ends. Download the results (or commit them to GitHub) before closing.

In [ ]:
from google.colab import files
!zip -qr results.zip results
files.download("results.zip")